# AmesChronicle：从深度数据诊断到高阶特征工程的完整实证研究

**研究主题：** 如何通过严谨的数据分析（Exploratory & Diagnostic Data Analysis）发现原始特征的缺陷与潜在非线性规律，有针对性地设计特征工程（Feature Engineering），并通过 4 阶段消融实验（Ablation Study）与 SHAP 归因量化每一项特征决策的真实收益。

---
## 核心研究脉络（The Causal Analytical Loop）
1. **数据怎么分析的？（Diagnostic Discovery）**：不盲目套公式，而是通过杠杆率诊断、缺失机制分层对比、偏态 Q-Q 检验、分层回归斜率拆解来定位问题。
2. **从分析中看出了哪些特征有用？（Hypothesis & Feature Design）**：
   - 发现单层面积（`1stFlrSF`, `2ndFlrSF`, `TotalBsmtSF`）被平房/双层户型割裂 $\rightarrow$ 构造物理总空间 `TotalSF`（相关性从 `0.63` 跃升至 `0.824`）。
   - 发现不同建筑品质（`OverallQual`）下每平方英尺的价格斜率相差 **2.67 倍**（`$52/sq.ft` vs `$139/sq.ft`） $\rightarrow$ 构造超线性乘积特征 `OverallQual_x_TotalSF`（相关性达 `0.865`，跃居全模型第 1）。
   - 发现房龄超 30 年的老房出现明显折旧分岔，近 12 年内翻新过的老房溢价 `+$34,000+` $\rightarrow$ 构造 `HouseAge`、`RemodAge` 与抗折旧指标 `QualityWeightedAge`。
   - 发现 28 个社区解释超过 `54%` 的房价方差，但小社区样本量不足 10 套 $\rightarrow$ 构造 5 折交叉经验贝叶斯平滑目标编码 `Neighborhood_TE`。
3. **加上这些特征后得到了什么结果？（Empirical Proof）**：
   - 5 折交叉验证 $R^2$ 从原始基线的 **`0.8501`** 跃升至 **`0.9272`**（线性模型 Ridge $R^2$ 更是从 `0.72` 跃升至 `0.91+`）。
   - 预测误差 `RMSE` 从 **`$30,275`** 降至 **`$21,102`**（误差大幅削减 **`-30.3%`**）。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
from src.data_loader import load_ames_raw
from src.features import (
    detect_leverage_outliers,
    impute_semantic_missing,
    transform_skewed_features,
    synthesize_domain_features,
    apply_oof_target_encoding,
    build_analytical_discovery_evidence,
)
from src.models import run_ablation_arena, compute_shap_and_simulator_weights

raw_df = load_ames_raw(Path('../data/raw'))
print(f'Loaded Ames Housing Dataset: {raw_df.shape[0]} rows × {raw_df.shape[1]} columns')

## 第一阶段：高杠杆离群点诊断与缺失值物理语义剖析

### 1.1 为什么要做离群点手术？（Cook's Distance 杠杆分析）
在绘制 `GrLivArea`（地上居住面积）与 `SalePrice` 的散点分布时，我们发现存在面积超过 `4,000 sq.ft` 但成交价却低于 `$200,000` 的极端反常样本（记录显示为 `Partial Sale` 未完工交易）。通过计算单变量回归的 **Cook's Distance ($D_i$)**，可以量化它们对回归斜率的扭曲程度。

In [ ]:
outlier_diag = detect_leverage_outliers(raw_df)
print('Detected High-Leverage Outlier Indices:', outlier_diag['outlier_indices'])
print(f"OLS R² Before Surgery: {outlier_diag['r2_before']:.4f} | After Surgery: {outlier_diag['r2_after']:.4f} (+{outlier_diag['r2_gain_pct']:.2f}%)")
print('Slope Before ($/sq.ft):', outlier_diag['regression_before']['slope'], '-> Slope After ($/sq.ft):', outlier_diag['regression_after']['slope'])

### 1.2 缺失值真的是“缺失”吗？（有设施 vs 无设施的房价分层检验）
传统做法往往将缺失率高于 50% 的列直接删除，或用众数填充。但我们通过计算**「字段非空 vs 字段为 NA 时的平均房价差（Facility Premium）」**发现：`FireplaceQu`、`BsmtQual`、`GarageType`、`PoolQC` 的 `NA` 代表**物理上没有壁炉/地下室/车库/泳池**，二者的平均房价差距高达 **`$65,000 ~ $88,000`**！

In [ ]:
clean_no_outliers = raw_df.drop(index=outlier_diag['outlier_indices']).reset_index(drop=True)
imputed_df, missing_report = impute_semantic_missing(clean_no_outliers)
pd.DataFrame(missing_report)[['feature', 'missing_pct', 'strategy', 'mean_price_present', 'mean_price_absent', 'facility_premium_usd']].head(12)

## 第二阶段：右偏长尾分布与 Box-Cox / Log1p 正态化修正

对 `SalePrice` 及核心连续特征进行偏度（Skewness $\gamma_1$）与峰度（Kurtosis $\gamma_2$）检验，发现原始房价呈严重右偏（$\gamma_1 > 1.7$），违反线性回归的高斯同方差假设。对其进行 `Log1p` 与 `Box-Cox` 变换后，偏度收敛至 `0` 附近。

In [ ]:
skew_df, skew_report = transform_skewed_features(imputed_df)
pd.DataFrame([
    {
        'Feature': k,
        'Raw Skew': v['raw_skew'],
        'Log1p Skew': v['log1p_skew'],
        'Box-Cox Skew': v['boxcox_skew'],
        'Box-Cox λ': v['boxcox_lambda'],
    }
    for k, v in skew_report.items()
])

## 第三阶段：从数据探索到领域特征合成（为什么构造这些新特征？）

我们对比每一个合成特征与其原始父特征（Parent Features）对房价的相关系数提升（$\Delta |r|$）：
1. **`OverallQual_x_TotalSF`**：捕捉品质与空间的乘性协同效应（$|r| = 0.865$）。
2. **`TotalSF`**：消除地下室、一楼、二楼的户型割裂（$|r|$ 从 `0.63` 升至 `0.824`）。
3. **`AmenityScore` & `TotalBath`**：聚合分散的精装序数指标与 4 个稀疏卫浴计数。
4. **`HouseAge`, `RemodAge`, `QualityWeightedAge`**：刻画房龄折旧与老房翻新溢价。

In [ ]:
synth_df, lineage_meta = synthesize_domain_features(skew_df)
pd.DataFrame(lineage_meta)[['feature', 'group', 'best_parent_corr', 'abs_corr', 'lift_over_parent', 'formula', 'empirical_result']]

## 第四阶段：28 个社区空间异质性与 5 折经验贝叶斯目标编码

使用 5-Fold Out-of-Fold Empirical Bayes Smoothing 计算 `Neighborhood_TE`，防止小样本社区过拟合：

In [ ]:
encoded_df, nb_profiles = apply_oof_target_encoding(synth_df)
pd.DataFrame(nb_profiles)[['neighborhood', 'tier', 'count', 'mean_price', 'smoothed_te', 'price_per_sqft', 'mean_qual']].head(10)

## 第五阶段：4 阶段消融实验（Ablation Arena）与 7 步增量收益验证

我们在完全相同的交叉验证划分下，对比 `Stage 0` 到 `Stage 3` 在 6 种主流模型上的表现，并拆解每一步特征工程带来的真实 RMSE 下降（美元）与 $R^2$ 增益：

In [ ]:
arena = run_ablation_arena(raw_df)
pd.DataFrame(arena['feature_stepwise_waterfall'])[['step', 'badge', 'delta_rmse_usd', 'rmse_after_usd', 'delta_r2', 'r2_after', 'analytical_reason']]